# ☁️ AI Dubbing Cloud GPU Server (v5.3)
**Run this on Colab → Your local IDE uses Colab's T4 GPU automatically!**

---
### How it works:
1. This notebook starts a GPU-powered API server on Colab
2. ngrok creates a public URL for it
3. You paste that URL in your local `.env` file
4. Your local `node backend/server.js` sends videos here for processing
5. Edge TTS / XTTS / ElevenLabs dubbing powered by Colab's 16GB VRAM!
---

## Step 1: Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"\n✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2: Install Dependencies (~3-5 min)

In [ ]:
# Install system dependencies
!sudo apt-get -y install espeak-ng libsndfile1-dev

# Install the community-maintained TTS fork (supports Python 3.12)
!pip install -q coqui-tts

# Install other dependencies
!pip install -q faster-whisper deep-translator ffmpeg-python edge-tts gtts demucs flask pyngrok
!pip install -q -U google-generativeai

# Install multi-engine dependencies (ElevenLabs + audio processing)
!pip install -q elevenlabs librosa soundfile

print("\n✅ All packages installed!")

## Step 3: Set Your ngrok Token
1. Go to [ngrok.com](https://ngrok.com) → Sign up (free)
2. Go to Dashboard → Your Authtoken
3. Paste it below

In [ ]:
NGROK_TOKEN = ""  # <-- PASTE YOUR NGROK TOKEN HERE

if not NGROK_TOKEN:
    print("❌ Please paste your ngrok token above!")
    print("   Get it free from: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    print("✅ Token set!")

## Step 4: Upload cloud_api.py
Upload the `cloud_api.py` file from your project's `python/` folder.
**Make sure it says v5.3 after upload!**

In [ ]:
import os
from google.colab import files

# Remove old file if it exists
if os.path.exists('/content/cloud_api.py'):
    os.remove('/content/cloud_api.py')
    print('Deleted old cloud_api.py')

print('Upload cloud_api.py from d:\\newProject\\python\\cloud_api.py:')
uploaded = files.upload()
print(f"\n✅ Uploaded: {list(uploaded.keys())}")

# Auto-verify correct version
with open('/content/cloud_api.py', 'r') as f:
    header = f.read(500)
if 'v5.3' in header:
    print('✅ VERSION OK: v5.3 detected!')
else:
    print('⚠️ WRONG FILE! v5.3 not found. Upload the latest cloud_api.py!')

## Step 5: Start the Cloud API Server 🚀
This will:
1. Pre-load XTTS model (~2 min first time)
2. Start Flask API on port 5050
3. Create ngrok tunnel
4. Print the **PUBLIC URL** you need to copy

In [ ]:
import subprocess
import threading
import time
from pyngrok import ngrok

# Kill existing ngrok processes to prevent 'too many tunnels' error
ngrok.kill()

# Set ngrok token
ngrok.set_auth_token(NGROK_TOKEN)

# Start ngrok tunnel
public_url = ngrok.connect(5050)
print("\n" + "="*60)
print(f"\ud83c\udf10 YOUR CLOUD API URL: {public_url}")
print("="*60)
print("\n\ud83d\udccb COPY THE URL ABOVE AND:")
print("   1. Create/edit file: d:\\newProject\\backend\\.env")
print(f"   2. Add this line: CLOUD_API_URL={public_url}")
print("   3. Restart your local server: node backend/server.js")
print("   4. Upload a video on localhost:5000 \u2192 It processes HERE on Colab GPU! \ud83d\ude80")
print("\n" + "="*60)
print("\u23f3 Starting Flask server...")
print("   Keep this tab open! Closing it stops the server.")
print("="*60 + "\n")

# Run Flask server (blocking)
!python cloud_api.py